# 개별종목 — RandomForest

## 실험 목적

KOSPI200 방향이 상승·보합·하락 중 어디인지 정해졌을 때, 같은 방향일 확률이 높은
개별종목을 찾기 위한 3분류 모델입니다.

후보는 매 거래일 **KOSPI 세부 업종지수 시가총액 상위 10개 × 업종별 KOSPI 보통주
시가총액 상위 5개**로 먼저 고정합니다. 업종의 미래 방향을 따로 예측하는 구조는 아닙니다.

## 공통 조건

| 항목 | 값 |
|---|---|
| 원천 | HF `full/daily_price_dev.parquet`, `full/index_price_dev.parquet` |
| 홀드아웃 | `20240901` 이후 접근 금지 |
| 라벨 | T일 판단 → T+1 `adj_open` 진입 → T+6 `adj_open` 평가, 종목 ±2% |
| 외부 검증 | 날짜 그룹 expanding 12폴드 |
| 최초 학습 | 750거래일 |
| 검증·gap | 폴드당 60거래일 · 직전 5거래일 제거 |
| class weight | 각 외부 폴드 내부에서 `None`과 `balanced` 재비교 |
| 선정 지표 | Accuracy·Macro F1·하락 Recall 조화평균 |

## OOS 결과

| Accuracy | Macro F1 | 하락 Recall | 핵심지표 조화평균 |
|---:|---:|---:|---:|
| 0.3822 | 0.3641 | 0.3388 | **0.3565** |

아래 셀은 저장된 실측 리포트에서 이 모델의 폴드 결과와 class weight 선택 횟수를 다시
읽습니다. 학습 구현은 `models/stock_experiment.py`, 피처·라벨은
`features/stock_model_dataset.py`가 정본입니다.


In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "reports" / "stock_model_experiment.json").exists():
    ROOT = ROOT.parent
report = json.loads((ROOT / "reports" / "stock_model_experiment.json").read_text(encoding="utf-8"))
model_name = 'RandomForest'

folds = pd.DataFrame(report["outer_fold_results"])
display(folds.loc[folds["model"].eq(model_name)].reset_index(drop=True))

weights = pd.DataFrame(report["selected_class_weight_counts"])
display(weights.loc[weights["model"].eq(model_name)].reset_index(drop=True))


,model,fold,selected_class_weight,train_dates,valid_dates,train_rows,valid_rows,train_end,valid_start,valid_end,accuracy,macro_f1,down_recall,core_harmonic_mean
0,RandomForest,1,NaN,750,60,36207,2880,20130401,20130409,20130704,0.365278,0.357634,0.286316,0.332357
1,RandomForest,2,balanced,999,60,48107,2880,20140403,20140411,20140710,0.433681,0.365556,0.237903,0.324507
2,RandomForest,3,balanced,1248,60,60029,2880,20150410,20150420,20150715,0.378125,0.377326,0.387370,0.380886
3,RandomForest,4,balanced,1497,60,72038,2973,20160414,20160422,20160719,0.380424,0.364850,0.295888,0.342890
4,RandomForest,5,balanced,1746,60,84118,2872,20170414,20170424,20170721,0.385446,0.347979,0.303625,0.342400
5,RandomForest,6,balanced,1995,60,96025,2940,20180424,20180503,20180731,0.369728,0.367641,0.340340,0.358724
6,RandomForest,7,balanced,2243,60,108142,2940,20190502,20190513,20190805,0.397279,0.354276,0.255319,0.324098
7,RandomForest,8,balanced,2492,60,120316,2936,20200507,20200515,20200806,0.347411,0.340092,0.519909,0.387486
8,RandomForest,9,balanced,2741,60,132514,2955,20210507,20210517,20210809,0.422335,0.395599,0.385311,0.400485
9,RandomForest,10,balanced,2990,60,144899,3000,20220511,20220519,20220812,0.347333,0.346793,0.315686,0.335933


,model,selected_class_weight,folds
0,RandomForest,None,1
1,RandomForest,balanced,11
